In [1]:

!pip -q install kagglehub

import kagglehub
import pandas as pd

path = kagglehub.dataset_download(
    "wordsforthewise/lending-club"
)

print("Dataset location:", path)


Using Colab cache for faster access to the 'lending-club' dataset.
Dataset location: /kaggle/input/lending-club


In [2]:

import os
import pandas as pd

# Path to the original Lending Club dataset
file_path = os.path.join(
    path,
    "accepted_2007_to_2018Q4.csv.gz"
)

# Define the loan outcomes for our project
selected_statuses = [
    "Fully Paid",
    "Charged Off",
    "Default"
]

# Read only the relevant completed loan outcomes
chunks = []
for chunk in pd.read_csv(
    file_path,
    compression="gzip",
    chunksize=100000,
    low_memory=False
):
    selected = chunk[
        chunk["loan_status"].isin(selected_statuses)
    ]
    chunks.append(selected)

# Combine the selected records
df = pd.concat(chunks, ignore_index=True)

print("Selected rows:", df.shape[0])
print("Total columns:", df.shape[1])

display(df.head())


Selected rows: 1345350
Total columns: 151


,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,68407277,NaN,3600.0,3600.0,3600.0,36 months,13.99,123.03,C,C4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,68355089,NaN,24700.0,24700.0,24700.0,36 months,11.99,820.28,C,C1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2,68341763,NaN,20000.0,20000.0,20000.0,60 months,10.78,432.66,B,B4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
3,68476807,NaN,10400.0,10400.0,10400.0,60 months,22.45,289.91,F,F1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
4,68426831,NaN,11950.0,11950.0,11950.0,36 months,13.44,405.18,C,C3,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:

# Stage 3.2 - Missing Values Assessment

import pandas as pd

missing_report = pd.DataFrame({
    "Missing Count": df.isnull().sum(),
    "Missing Percentage": (
        df.isnull().mean() * 100
    ).round(2)
})

missing_report = missing_report.sort_values(
    by="Missing Percentage",
    ascending=False
)

print("Total columns:", len(df.columns))

print(
    "Columns with missing values:",
    (missing_report["Missing Count"] > 0).sum()
)

display(missing_report.head(30))


Total columns: 151
Columns with missing values: 105


,Missing Count,Missing Percentage
member_id,1345350,100.00
next_pymnt_d,1345310,100.00
orig_projected_additional_accrued_interest,1341589,99.72
hardship_payoff_balance_amount,1339594,99.57
hardship_last_payment_amount,1339594,99.57
hardship_reason,1339594,99.57
hardship_type,1339594,99.57
hardship_status,1339594,99.57
hardship_amount,1339594,99.57
deferral_term,1339594,99.57


In [4]:

# Count columns by missing-value percentage

print(
    "Columns with 100% missing:",
    (missing_report["Missing Percentage"] == 100).sum()
)

print(
    "Columns with more than 80% missing:",
    (missing_report["Missing Percentage"] > 80).sum()
)

print(
    "Columns with more than 50% missing:",
    (missing_report["Missing Percentage"] > 50).sum()
)

print(
    "Columns with 0% missing:",
    (missing_report["Missing Count"] == 0).sum()
)


Columns with 100% missing: 2
Columns with more than 80% missing: 40
Columns with more than 50% missing: 58
Columns with 0% missing: 46


In [5]:

# Show columns with more than 80% missing values

high_missing = missing_report[
    missing_report["Missing Percentage"] > 80
]

display(high_missing)

print(
    "Number of high-missing columns:",
    len(high_missing)
)


,Missing Count,Missing Percentage
member_id,1345350,100.00
next_pymnt_d,1345310,100.00
orig_projected_additional_accrued_interest,1341589,99.72
hardship_payoff_balance_amount,1339594,99.57
hardship_last_payment_amount,1339594,99.57
hardship_reason,1339594,99.57
hardship_type,1339594,99.57
hardship_status,1339594,99.57
hardship_amount,1339594,99.57
deferral_term,1339594,99.57


Number of high-missing columns: 40


In [6]:

# Stage 3.3 - Check duplicate records

print("Total rows:", len(df))

print("Fully duplicated rows:", df.duplicated().sum())

print("Duplicated loan IDs:", df["id"].duplicated().sum())

print("Missing loan IDs:", df["id"].isna().sum())


Total rows: 1345350
Fully duplicated rows: 0
Duplicated loan IDs: 0
Missing loan IDs: 0


In [7]:

# Stage 3.4 - Check data types

print("Data type summary:")
print(df.dtypes.value_counts())

print("\nExamples of column data types:")
display(
    df[
        [
            "loan_amnt",
            "term",
            "int_rate",
            "emp_length",
            "annual_inc",
            "earliest_cr_line",
            "loan_status"
        ]
    ].dtypes
)


Data type summary:
float64    113
object      38
Name: count, dtype: int64

Examples of column data types:


,0
loan_amnt,float64
term,object
int_rate,float64
emp_length,object
annual_inc,float64
earliest_cr_line,object
loan_status,object


In [8]:

# Stage 3.4 - Inspect text values before cleaning

print("Loan term values:")
print(df["term"].value_counts(dropna=False))

print("\nEmployment length values:")
print(df["emp_length"].value_counts(dropna=False))

print("\nEarliest credit line examples:")
print(df["earliest_cr_line"].dropna().head(10))


Loan term values:
term
36 months    1020768
60 months     324582
Name: count, dtype: int64

Employment length values:
emp_length
10+ years    442209
2 years      121751
< 1 year     108065
3 years      107602
1 year        88495
5 years       84154
4 years       80558
NaN           78516
6 years       62735
8 years       60704
7 years       59624
9 years       50937
Name: count, dtype: int64

Earliest credit line examples:
0    Aug-2003
1    Dec-1999
2    Aug-2000
3    Jun-1998
4    Oct-1987
5    Jun-1990
6    Feb-1999
7    Apr-2002
8    Nov-1994
9    Jun-1996
Name: earliest_cr_line, dtype: object


In [9]:

# Stage 3.4 - Validate credit history dates

parsed_dates = pd.to_datetime(
    df["earliest_cr_line"],
    format="%b-%Y",
    errors="coerce"
)

print(
    "Missing original dates:",
    df["earliest_cr_line"].isna().sum()
)

print(
    "Invalid non-missing dates:",
    (
        df["earliest_cr_line"].notna()
        & parsed_dates.isna()
    ).sum()
)

print("Earliest date:", parsed_dates.min())
print("Latest date:", parsed_dates.max())


Missing original dates: 0
Invalid non-missing dates: 0
Earliest date: 1934-04-01 00:00:00
Latest date: 2015-10-01 00:00:00


In [10]:

# Check whether credit history starts after loan issue date

issue_dates = pd.to_datetime(
    df["issue_d"],
    format="%b-%Y",
    errors="coerce"
)

invalid_order = parsed_dates > issue_dates

print("Missing issue dates:", df["issue_d"].isna().sum())

print(
    "Invalid issue date formats:",
    (df["issue_d"].notna() & issue_dates.isna()).sum()
)

print(
    "Credit history dates after loan issue:",
    invalid_order.sum()
)



Missing issue dates: 0
Invalid issue date formats: 0
Credit history dates after loan issue: 0


In [11]:

# Stage 3.5 - Initial Outlier Inspection

numeric_columns = [
    "loan_amnt",
    "annual_inc",
    "dti",
    "int_rate",
    "revol_util"
]

display(
    df[numeric_columns].describe().T[
        ["min", "25%", "50%", "75%", "max"]
    ]
)


,min,25%,50%,75%,max
loan_amnt,500.00,8000.00,12000.00,20000.00,40000.00
annual_inc,0.00,45780.00,65000.00,90000.00,10999200.00
dti,-1.00,11.79,17.61,24.06,999.00
int_rate,5.31,9.75,12.74,15.99,30.99
revol_util,0.00,33.40,52.20,70.70,892.30


In [12]:

# Stage 3.5 - IQR Outlier Detection

numeric_columns = [
    "loan_amnt",
    "annual_inc",
    "dti",
    "int_rate",
    "revol_util"
]

for col in numeric_columns:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    outliers = (
        (df[col] < lower_bound) |
        (df[col] > upper_bound)
    )

    print(
        f"{col}: {outliers.sum():,} potential outliers "
        f"(lower={lower_bound:.2f}, upper={upper_bound:.2f})"
    )


loan_amnt: 7,159 potential outliers (lower=-10000.00, upper=38000.00)
annual_inc: 65,873 potential outliers (lower=-20550.00, upper=156330.00)
dti: 5,473 potential outliers (lower=-6.62, upper=42.47)
int_rate: 24,981 potential outliers (lower=0.39, upper=25.35)
revol_util: 72 potential outliers (lower=-22.55, upper=126.65)


In [13]:

# Stage 3.5 - Check potentially problematic values

print(
    "Negative DTI values:",
    (df["dti"] < 0).sum()
)

print(
    "DTI values equal to 999:",
    (df["dti"] == 999).sum()
)

print(
    "Revol_util values above 100:",
    (df["revol_util"] > 100).sum()
)

print(
    "Annual income equal to zero:",
    (df["annual_inc"] == 0).sum()
)


Negative DTI values: 2
DTI values equal to 999: 38
Revol_util values above 100: 4687
Annual income equal to zero: 361


In [14]:

# Stage 3.6 - Select relevant columns for cleaning

candidate_features = [
    "loan_amnt", "term", "int_rate", "installment",
    "grade", "sub_grade", "emp_length", "home_ownership",
    "annual_inc", "verification_status", "purpose",
    "addr_state", "dti", "delinq_2yrs", "earliest_cr_line",
    "fico_range_low", "fico_range_high", "inq_last_6mths",
    "open_acc", "pub_rec", "revol_bal", "revol_util",
    "total_acc", "mort_acc", "pub_rec_bankruptcies"
]

selected_columns = ["id", "loan_status"] + candidate_features

df_clean = df[selected_columns].copy()

print("Rows in working dataset:", df_clean.shape[0])
print("Columns in working dataset:", df_clean.shape[1])
print("Missing columns:", set(selected_columns) - set(df.columns))


Rows in working dataset: 1345350
Columns in working dataset: 27
Missing columns: set()


In [15]:

# Stage 3.6 - Missing values in selected columns

selected_missing = pd.DataFrame({
    "Missing Count": df_clean.isna().sum(),
    "Missing Percentage": (
        df_clean.isna().mean() * 100
    ).round(2)
})

selected_missing = selected_missing[
    selected_missing["Missing Count"] > 0
].sort_values(
    by="Missing Count",
    ascending=False
)

display(selected_missing)

print(
    "Columns with missing values:",
    len(selected_missing)
)


,Missing Count,Missing Percentage
emp_length,78516,5.84
mort_acc,47281,3.51
revol_util,857,0.06
pub_rec_bankruptcies,697,0.05
dti,374,0.03
inq_last_6mths,1,0.00


Columns with missing values: 6


In [16]:

# Stage 3.6 - Handle missing employment length

print(
    "Missing emp_length before:",
    df_clean["emp_length"].isna().sum()
)

df_clean["emp_length"] = (
    df_clean["emp_length"].fillna("Unknown")
)

print(
    "Missing emp_length after:",
    df_clean["emp_length"].isna().sum()
)

print(
    "Unknown employment length:",
    (df_clean["emp_length"] == "Unknown").sum()
)


Missing emp_length before: 78516
Missing emp_length after: 0
Unknown employment length: 78516


In [17]:

# Stage 3.6 - Inspect median values before imputation

numeric_missing_columns = [
    "mort_acc",
    "revol_util",
    "pub_rec_bankruptcies",
    "dti",
    "inq_last_6mths"
]

display(
    df_clean[numeric_missing_columns]
    .median()
    .to_frame(name="Median")
)


,Median
mort_acc,1.00
revol_util,52.20
pub_rec_bankruptcies,0.00
dti,17.61
inq_last_6mths,0.00


In [18]:

# Stage 3.6 - Check remaining missing values

print("Rows:", df_clean.shape[0])
print("Columns:", df_clean.shape[1])

print("\nRemaining missing values by column:")
print(df_clean.isna().sum()[df_clean.isna().sum() > 0])

print(
    "\nTotal remaining missing cells:",
    df_clean.isna().sum().sum()
)


Rows: 1345350
Columns: 27

Remaining missing values by column:
dti                       374
inq_last_6mths              1
revol_util                857
mort_acc                47281
pub_rec_bankruptcies      697
dtype: int64

Total remaining missing cells: 49210


In [19]:

# Create binary target for loan default prediction

target_mapping = {
    "Fully Paid": 0,
    "Charged Off": 1,
    "Default": 1
}

df_clean["target"] = df_clean["loan_status"].map(
    target_mapping
)

print("Target distribution:")
print(df_clean["target"].value_counts())

print(
    "\nMissing target values:",
    df_clean["target"].isna().sum()
)


Target distribution:
target
0    1076751
1     268599
Name: count, dtype: int64

Missing target values: 0


In [20]:

# Stage 3.6 - Split data before numerical imputation

from sklearn.model_selection import train_test_split

# Keep loan IDs and original status out of the model features
X = df_clean.drop(
    columns=["id", "loan_status", "target"]
)

y = df_clean["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

print("\nTraining target distribution:")
print(y_train.value_counts())

print("\nTesting target distribution:")
print(y_test.value_counts())


Training rows: 1076280
Testing rows: 269070

Training target distribution:
target
0    861401
1    214879
Name: count, dtype: int64

Testing target distribution:
target
0    215350
1     53720
Name: count, dtype: int64


In [21]:

# Stage 3.6 - Median imputation without data leakage

from sklearn.impute import SimpleImputer

numeric_missing_columns = [
    "mort_acc",
    "revol_util",
    "pub_rec_bankruptcies",
    "dti",
    "inq_last_6mths"
]

# Keep the original train/test data unchanged
X_train_clean = X_train.copy()
X_test_clean = X_test.copy()

# Learn median values from training data only
imputer = SimpleImputer(strategy="median")

X_train_clean[numeric_missing_columns] = imputer.fit_transform(
    X_train[numeric_missing_columns]
)

# Apply training medians to testing data
X_test_clean[numeric_missing_columns] = imputer.transform(
    X_test[numeric_missing_columns]
)

print(
    "Training missing values after:",
    X_train_clean.isna().sum().sum()
)

print(
    "Testing missing values after:",
    X_test_clean.isna().sum().sum()
)

print("\nTraining medians used:")
print(
    pd.Series(
        imputer.statistics_,
        index=numeric_missing_columns
    )
)


Training missing values after: 0
Testing missing values after: 0

Training medians used:
mort_acc                 1.00
revol_util              52.20
pub_rec_bankruptcies     0.00
dti                     17.62
inq_last_6mths           0.00
dtype: float64


In [22]:

# Compare missing values before and after cleaning

before_missing = (
    X_train.isna().sum().sum()
    + X_test.isna().sum().sum()
)

after_missing = (
    X_train_clean.isna().sum().sum()
    + X_test_clean.isna().sum().sum()
)

print("Missing cells before numerical imputation:", before_missing)
print("Missing cells after numerical imputation:", after_missing)

print("Total rows retained:", len(X_train_clean) + len(X_test_clean))
print("Number of model features:", X_train_clean.shape[1])


Missing cells before numerical imputation: 49210
Missing cells after numerical imputation: 0
Total rows retained: 1345350
Number of model features: 25


In [23]:

# Inspect negative DTI values

print("Negative DTI values:")
print(df_clean.loc[df_clean["dti"] < 0, "dti"].value_counts())

print("\nNumber of affected records:")
print((df_clean["dti"] < 0).sum())


Negative DTI values:
dti
-1.0    2
Name: count, dtype: int64

Number of affected records:
2


In [24]:

# Check negative DTI values in train and test sets

print(
    "Negative DTI in training:",
    (X_train_clean["dti"] < 0).sum()
)

print(
    "Negative DTI in testing:",
    (X_test_clean["dti"] < 0).sum()
)


Negative DTI in training: 1
Negative DTI in testing: 1


In [25]:

# Replace negative DTI values using the training median

# Median from training data
dti_median = imputer.statistics_[
    numeric_missing_columns.index("dti")
]

# Replace negative values in both datasets
X_train_clean.loc[
    X_train_clean["dti"] < 0, "dti"
] = dti_median

X_test_clean.loc[
    X_test_clean["dti"] < 0, "dti"
] = dti_median

print("DTI median used:", dti_median)

print(
    "Negative DTI in training after:",
    (X_train_clean["dti"] < 0).sum()
)

print(
    "Negative DTI in testing after:",
    (X_test_clean["dti"] < 0).sum()
)


DTI median used: 17.62
Negative DTI in training after: 0
Negative DTI in testing after: 0


In [26]:

# Check DTI values equal to 999

print(
    "DTI = 999 in training:",
    (X_train_clean["dti"] == 999).sum()
)

print(
    "DTI = 999 in testing:",
    (X_test_clean["dti"] == 999).sum()
)

print(
    "Total DTI = 999:",
    (X_train_clean["dti"] == 999).sum()
    + (X_test_clean["dti"] == 999).sum()
)


DTI = 999 in training: 32
DTI = 999 in testing: 6
Total DTI = 999: 38


In [27]:

# Inspect unusually high DTI values

print("DTI values above 100:")
print((df["dti"] > 100).sum())

print("\nMost common DTI values above 100:")
print(
    df.loc[df["dti"] > 100, "dti"]
    .value_counts()
    .head(10)
)


DTI values above 100:
533

Most common DTI values above 100:
dti
999.00    38
107.84     2
101.20     2
101.50     2
105.38     2
104.00     2
163.25     2
149.46     2
117.71     2
102.73     2
Name: count, dtype: int64


In [28]:

# Inspect income for loans with DTI = 999

dti_999 = df.loc[
    df["dti"] == 999,
    ["dti", "annual_inc", "loan_status"]
]

print("Records with DTI = 999:", len(dti_999))

print(
    "Zero annual income:",
    (dti_999["annual_inc"] == 0).sum()
)

print("\nAnnual income summary:")
display(dti_999["annual_inc"].describe())


Records with DTI = 999: 38
Zero annual income: 0

Annual income summary:


,annual_inc
count,38.000000
mean,871.894737
std,658.299931
min,16.000000
25%,421.750000
50%,1000.000000
75%,1000.000000
max,3000.000000


In [29]:

# Check how DTI = 999 is distributed by loan issue year

dti_999_check = df.loc[
    df["dti"] == 999,
    ["issue_d", "annual_inc"]
].copy()

dti_999_check["issue_year"] = pd.to_datetime(
    dti_999_check["issue_d"],
    format="%b-%Y"
).dt.year

print("DTI = 999 by loan issue year:")
print(
    dti_999_check["issue_year"]
    .value_counts()
    .sort_index()
)



DTI = 999 by loan issue year:
issue_year
2015     1
2016     6
2017    25
2018     6
Name: count, dtype: int64


In [30]:

# Inspect high revolving credit utilization values

print("Revol_util values above 100:")
print((df["revol_util"] > 100).sum())

print("\nRevol_util values above 200:")
print((df["revol_util"] > 200).sum())

print("\nRevol_util values above 500:")
print((df["revol_util"] > 500).sum())

print("\nHighest 10 revol_util values:")
print(df["revol_util"].nlargest(10).to_list())


Revol_util values above 100:
4687

Revol_util values above 200:
2

Revol_util values above 500:
1

Highest 10 revol_util values:
[892.3, 366.6, 193.0, 184.6, 182.8, 180.3, 177.7, 172.0, 165.8, 162.0]


In [31]:

# Inspect records with very high revolving utilization

high_revol = df.loc[
    df["revol_util"] > 200,
    [
        "revol_util",
        "revol_bal",
        "annual_inc",
        "loan_status"
    ]
]

display(high_revol)


,revol_util,revol_bal,annual_inc,loan_status
799628,366.6,2566.0,48000.0,Charged Off
835912,892.3,2677.0,45000.0,Fully Paid


In [32]:

# Inspect implied revolving credit limits

high_revol_check = high_revol.copy()

high_revol_check["implied_credit_limit"] = (
    high_revol_check["revol_bal"]
    / (high_revol_check["revol_util"] / 100)
)

display(
    high_revol_check[
        ["revol_util", "revol_bal", "implied_credit_limit"]
    ].round(2)
)


,revol_util,revol_bal,implied_credit_limit
799628,366.6,2566.0,699.95
835912,892.3,2677.0,300.01


In [33]:

# Stage 3.6 - Convert loan term to numeric months

print("Before conversion:")
print(X_train_clean["term"].value_counts())

for data in [X_train_clean, X_test_clean]:
    data["term"] = pd.to_numeric(
        data["term"].str.strip().str.replace(
            " months", "", regex=False
        ),
        errors="coerce"
    )

print("\nAfter conversion:")
print(X_train_clean["term"].value_counts())

print(
    "\nMissing term values after conversion:",
    X_train_clean["term"].isna().sum()
    + X_test_clean["term"].isna().sum()
)


Before conversion:
term
36 months    816667
60 months    259613
Name: count, dtype: int64

After conversion:
term
36    816667
60    259613
Name: count, dtype: int64

Missing term values after conversion: 0


In [34]:

# Stage 3.6 - Convert credit history dates

for data in [X_train_clean, X_test_clean]:
    data["earliest_cr_line"] = pd.to_datetime(
        data["earliest_cr_line"],
        format="%b-%Y",
        errors="coerce"
    )

print(
    "Training data type:",
    X_train_clean["earliest_cr_line"].dtype
)

print(
    "Testing data type:",
    X_test_clean["earliest_cr_line"].dtype
)

print(
    "Missing dates after conversion:",
    X_train_clean["earliest_cr_line"].isna().sum()
    + X_test_clean["earliest_cr_line"].isna().sum()
)


Training data type: datetime64[ns]
Testing data type: datetime64[ns]
Missing dates after conversion: 0


In [35]:

# Check availability of loan issue dates

print("issue_d in original dataset:", "issue_d" in df.columns)

print(
    "issue_d in training features:",
    "issue_d" in X_train_clean.columns
)

print(
    "issue_d in testing features:",
    "issue_d" in X_test_clean.columns
)


issue_d in original dataset: True
issue_d in training features: False
issue_d in testing features: False


In [36]:

# Get loan issue dates for training and testing

train_issue_dates = pd.to_datetime(
    df_clean.loc[X_train_clean.index, "id"].map(
        df.set_index("id")["issue_d"]
    ),
    format="%b-%Y",
    errors="coerce"
)

test_issue_dates = pd.to_datetime(
    df_clean.loc[X_test_clean.index, "id"].map(
        df.set_index("id")["issue_d"]
    ),
    format="%b-%Y",
    errors="coerce"
)

print("Training issue dates:", len(train_issue_dates))
print("Testing issue dates:", len(test_issue_dates))

print(
    "Missing issue dates:",
    train_issue_dates.isna().sum()
    + test_issue_dates.isna().sum()
)


Training issue dates: 1076280
Testing issue dates: 269070
Missing issue dates: 0


In [37]:

# Stage 3.6 - Create credit history length feature

# Calculate the difference in months
for data, issue_dates in [
    (X_train_clean, train_issue_dates),
    (X_test_clean, test_issue_dates)
]:
    credit_dates = data["earliest_cr_line"]

    data["credit_history_months"] = (
        (issue_dates.dt.year - credit_dates.dt.year) * 12
        + (issue_dates.dt.month - credit_dates.dt.month)
    )

print(
    "Training credit history months:",
    X_train_clean["credit_history_months"].describe()
)

print(
    "\nTesting credit history months:",
    X_test_clean["credit_history_months"].describe()
)

print(
    "\nNegative credit history lengths:",
    (X_train_clean["credit_history_months"] < 0).sum()
    + (X_test_clean["credit_history_months"] < 0).sum()
)

print(
    "Missing credit history lengths:",
    X_train_clean["credit_history_months"].isna().sum()
    + X_test_clean["credit_history_months"].isna().sum()
)


Training credit history months: count    1.076280e+06
mean     1.950480e+02
std      9.005559e+01
min      3.600000e+01
25%      1.350000e+02
50%      1.770000e+02
75%      2.400000e+02
max      9.990000e+02
Name: credit_history_months, dtype: float64

Testing credit history months: count    269070.000000
mean        195.368737
std          90.142302
min          36.000000
25%         135.000000
50%         177.000000
75%         241.000000
max         804.000000
Name: credit_history_months, dtype: float64

Negative credit history lengths: 0
Missing credit history lengths: 0


In [38]:

# Remove raw date after creating the numeric feature

X_train_clean = X_train_clean.drop(
    columns=["earliest_cr_line"]
)

X_test_clean = X_test_clean.drop(
    columns=["earliest_cr_line"]
)

print("Training features:", X_train_clean.shape[1])
print("Testing features:", X_test_clean.shape[1])

print(
    "Credit history feature available:",
    "credit_history_months" in X_train_clean.columns
    and "credit_history_months" in X_test_clean.columns
)

print(
    "Raw date removed:",
    "earliest_cr_line" not in X_train_clean.columns
    and "earliest_cr_line" not in X_test_clean.columns
)


Training features: 25
Testing features: 25
Credit history feature available: True
Raw date removed: True


In [39]:

# Stage 3 - Final data quality check

print("Training rows:", X_train_clean.shape[0])
print("Testing rows:", X_test_clean.shape[0])

print("\nNumber of features:", X_train_clean.shape[1])

print(
    "\nMissing values in training:",
    X_train_clean.isna().sum().sum()
)

print(
    "Missing values in testing:",
    X_test_clean.isna().sum().sum()
)

print("\nData types:")
print(X_train_clean.dtypes.value_counts())

print("\nObject columns:")
print(X_train_clean.select_dtypes(include="object").columns.tolist())


Training rows: 1076280
Testing rows: 269070

Number of features: 25

Missing values in training: 0
Missing values in testing: 0

Data types:
float64    16
object      7
int64       2
Name: count, dtype: int64

Object columns:
['grade', 'sub_grade', 'emp_length', 'home_ownership', 'verification_status', 'purpose', 'addr_state']


In [40]:

# Check categorical features before encoding

categorical_columns = (
    X_train_clean
    .select_dtypes(include="object")
    .columns
)

for col in categorical_columns:
    print(
        f"{col}:",
        X_train_clean[col].nunique(),
        "unique categories"
    )


grade: 7 unique categories
sub_grade: 35 unique categories
emp_length: 12 unique categories
home_ownership: 6 unique categories
verification_status: 3 unique categories
purpose: 14 unique categories
addr_state: 51 unique categories


In [41]:

# Stage 3.4 - Inspect categorical values

for col in [
    "grade",
    "emp_length",
    "home_ownership",
    "verification_status"
]:
    print(f"\n{col}:")
    print(
        X_train_clean[col]
        .value_counts(dropna=False)
    )



grade:
grade
B    314111
C    305211
A    187981
D    161005
E     74981
F     25637
G      7354
Name: count, dtype: int64

emp_length:
emp_length
10+ years    353596
2 years       97361
< 1 year      86589
3 years       86313
1 year        70768
5 years       67281
4 years       64433
Unknown       62819
6 years       50117
8 years       48696
7 years       47770
9 years       40537
Name: count, dtype: int64

home_ownership:
home_ownership
MORTGAGE    532418
RENT        427804
OWN         115676
ANY            227
OTHER          116
NONE            39
Name: count, dtype: int64

verification_status:
verification_status
Source Verified    417079
Verified           334493
Not Verified       324708
Name: count, dtype: int64


In [42]:

# Inspect remaining categorical values

for col in ["sub_grade", "purpose", "addr_state"]:
    print(f"\n{col}:")
    print(sorted(X_train_clean[col].dropna().unique()))



sub_grade:
['A1', 'A2', 'A3', 'A4', 'A5', 'B1', 'B2', 'B3', 'B4', 'B5', 'C1', 'C2', 'C3', 'C4', 'C5', 'D1', 'D2', 'D3', 'D4', 'D5', 'E1', 'E2', 'E3', 'E4', 'E5', 'F1', 'F2', 'F3', 'F4', 'F5', 'G1', 'G2', 'G3', 'G4', 'G5']

purpose:
['car', 'credit_card', 'debt_consolidation', 'educational', 'home_improvement', 'house', 'major_purchase', 'medical', 'moving', 'other', 'renewable_energy', 'small_business', 'vacation', 'wedding']

addr_state:
['AK', 'AL', 'AR', 'AZ', 'CA', 'CO', 'CT', 'DC', 'DE', 'FL', 'GA', 'HI', 'IA', 'ID', 'IL', 'IN', 'KS', 'KY', 'LA', 'MA', 'MD', 'ME', 'MI', 'MN', 'MO', 'MS', 'MT', 'NC', 'ND', 'NE', 'NH', 'NJ', 'NM', 'NV', 'NY', 'OH', 'OK', 'OR', 'PA', 'RI', 'SC', 'SD', 'TN', 'TX', 'UT', 'VA', 'VT', 'WA', 'WI', 'WV', 'WY']


In [43]:

# Review features before machine learning

print("Current model features:\n")

for i, column in enumerate(X_train_clean.columns, start=1):
    print(f"{i}. {column}")

print("\nTotal features:", X_train_clean.shape[1])


Current model features:

1. loan_amnt
2. term
3. int_rate
4. installment
5. grade
6. sub_grade
7. emp_length
8. home_ownership
9. annual_inc
10. verification_status
11. purpose
12. addr_state
13. dti
14. delinq_2yrs
15. fico_range_low
16. fico_range_high
17. inq_last_6mths
18. open_acc
19. pub_rec
20. revol_bal
21. revol_util
22. total_acc
23. mort_acc
24. pub_rec_bankruptcies
25. credit_history_months

Total features: 25


In [44]:

# Check relationship between grade and sub_grade

grade_mismatch = (
    X_train_clean["grade"]
    != X_train_clean["sub_grade"].str[0]
)

print(
    "Grade/sub-grade mismatches:",
    grade_mismatch.sum()
)

print(
    "Matching records:",
    (~grade_mismatch).sum()
)


Grade/sub-grade mismatches: 0
Matching records: 1076280


In [45]:

# Remove redundant grade feature

X_train_clean = X_train_clean.drop(columns=["grade"])
X_test_clean = X_test_clean.drop(columns=["grade"])

print("Training features:", X_train_clean.shape[1])
print("Testing features:", X_test_clean.shape[1])

print(
    "Grade removed:",
    "grade" not in X_train_clean.columns
    and "grade" not in X_test_clean.columns
)

print(
    "Sub-grade retained:",
    "sub_grade" in X_train_clean.columns
    and "sub_grade" in X_test_clean.columns
)


Training features: 24
Testing features: 24
Grade removed: True
Sub-grade retained: True


In [46]:

# Final check after feature removal

print("Training rows:", X_train_clean.shape[0])
print("Testing rows:", X_test_clean.shape[0])

print("Training features:", X_train_clean.shape[1])
print("Testing features:", X_test_clean.shape[1])

print(
    "Total missing values:",
    X_train_clean.isna().sum().sum()
    + X_test_clean.isna().sum().sum()
)

print(
    "Categorical features:",
    X_train_clean.select_dtypes(
        include="object"
    ).columns.tolist()
)


Training rows: 1076280
Testing rows: 269070
Training features: 24
Testing features: 24
Total missing values: 0
Categorical features: ['sub_grade', 'emp_length', 'home_ownership', 'verification_status', 'purpose', 'addr_state']


In [47]:

# Check memory usage before saving cleaned data

train_memory = (
    X_train_clean.memory_usage(deep=True).sum()
    / (1024 ** 2)
)

test_memory = (
    X_test_clean.memory_usage(deep=True).sum()
    / (1024 ** 2)
)

print(f"Training data size: {train_memory:.2f} MB")
print(f"Testing data size: {test_memory:.2f} MB")
print(f"Total size: {train_memory + test_memory:.2f} MB")


Training data size: 503.33 MB
Testing data size: 124.81 MB
Total size: 628.14 MB


In [48]:

# Save cleaned training and testing data as Parquet

import os

output_dir = "/content/cleaned_data"
os.makedirs(output_dir, exist_ok=True)

train_data = X_train_clean.copy()
test_data = X_test_clean.copy()

train_data["target"] = y_train
test_data["target"] = y_test

train_path = os.path.join(output_dir, "lending_club_train.parquet")
test_path = os.path.join(output_dir, "lending_club_test.parquet")

train_data.to_parquet(train_path, index=False)
test_data.to_parquet(test_path, index=False)

print("Training file saved:", os.path.exists(train_path))
print("Testing file saved:", os.path.exists(test_path))

print("\nSaved files:")
print(os.listdir(output_dir))


Training file saved: True
Testing file saved: True

Saved files:
['lending_club_test.parquet', 'lending_club_train.parquet']


In [50]:

# Save cleaned datasets permanently in Google Drive

from google.colab import drive
import os
import shutil

drive.mount("/content/drive")

drive_folder = "/content/drive/MyDrive/LendingClub_Project/cleaned_data"

os.makedirs(drive_folder, exist_ok=True)

shutil.copy2(
    train_path,
    os.path.join(drive_folder, "lending_club_train.parquet")
)

shutil.copy2(
    test_path,
    os.path.join(drive_folder, "lending_club_test.parquet")
)

print("Files saved in Google Drive:")
print(os.listdir(drive_folder))


Mounted at /content/drive
Files saved in Google Drive:
['lending_club_train.parquet', 'lending_club_test.parquet']
